Dado que el entrenamiento de redes neuronales es una tarea  muy costosa, **se recomienda ejecutar el notebooks en [Google Colab](https://colab.research.google.com)**, por supuesto también se puede ejecutar en local.

Al entrar en [Google Colab](https://colab.research.google.com) bastará con hacer click en `upload` y subir este notebook. No olvide luego descargarlo en `File->Download .ipynb`

**El examen deberá ser entregado con las celdas ejecutadas, si alguna celda no está ejecutadas no se contará.**

El examen se divide en tres partes, con la puntuación que se indica a continuación. La puntuación máxima será 10.

    
- [Actividad 1: Redes Recurrentes](#actividad_1): 10 pts
    - [Cuestión 1](#3.1): 2.5 pt
    - [Cuestión 2](#3.2): 2.5 pt
    - [Cuestión 3](#3.3): 2.5 pts
    - [Cuestión 4](#3.4): 1.25 pts
    - [Cuestión 5](#3.5): 1.25 pts



In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

I0000 00:00:1794421290.967302   49308 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


<a name='actividad_1'></a>
# Actividad 1: Redes Recurrentes


- [Cuestión 1](#3.1): 2.5 pt
- [Cuestión 2](#3.2): 2.5 pt
- [Cuestión 3](#3.3): 2.5 pts
- [Cuestión 4](#3.4): 1.25 pts
- [Cuestión 5](#3.5): 1.25 pts

Vamos a usar un dataset de las temperaturas mínimas diarias en Melbourne. La tarea será la de predecir la temperatura mínima en dos días. Puedes usar técnicas de series temporales vistas en otras asignaturas, pero no es necesario.


In [2]:
dataset_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
data_dir = tf.keras.utils.get_file('daily-min-temperatures.csv', origin=dataset_url)

In [3]:
df = pd.read_csv(data_dir, parse_dates=['Date'])
df.head()

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [4]:
temperatures = df['Temp'].values
print('number of samples:', len(temperatures))
train_data = temperatures[:3000]
test_data = temperatures[3000:]
print('number of train samples:', len(train_data))
print('number of test samples:', len(test_data))
print('firsts trainn samples:', train_data[:10])

number of samples: 3650
number of train samples: 3000
number of test samples: 650
firsts trainn samples: [20.7 17.9 18.8 14.6 15.8 15.8 15.8 17.4 21.8 20. ]


<a name='3.1'></a>
## Cuestión 1: Convierta `train_data` y `test_data`  en ventanas de tamaño 5, para predecir el valor en 2 días

En la nomenclatura de [Introduction_to_RNN_Time_Series.ipynb](https://github.com/ezponda/intro_deep_learning/blob/main/class/RNN/Introduction_to_RNN_Time_Series.ipynb)
```python
past, future = (5, 2)
```

Para las primeras 10 muestras de train_data `[20.7, 17.9, 18.8, 14.6, 15.8, 15.8, 15.8, 17.4, 21.8, 20. ]` el resultado debería ser:

```python
x[0] : [20.7, 17.9, 18.8, 14.6, 15.8] , y[0]: 15.8
x[1] : [17.9, 18.8, 14.6, 15.8, 15.8] , y[1]: 17.4
x[2] : [18.8, 14.6, 15.8, 15.8, 15.8] , y[2]: 21.8
x[3] : [14.6, 15.8, 15.8, 15.8, 17.4] , y[3]: 20.             
```

In [5]:
def create_windows_np(data, window_size, horizon, shuffle=False):
    """
    Creates a dataset from the given time series data using NumPy.
    
    Parameters:
    data (np.ndarray): Time series data with one dimension.
    window_size (int): The number of past time steps to use as input features.
    horizon (int): The number of future time steps to predict.
    shuffle (bool): Shuffle the windows or not.
    
    Returns:
    tuple: A tuple containing the input-output pairs (windows, targets) as NumPy arrays.
    """

    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size+horizon-1])

    X, y = np.array(X), np.array(y)
    
    if shuffle:
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]
    
    return X, y

In [6]:
past, future = (5, 2)
X_train, y_train = create_windows_np(train_data, window_size=past, horizon=future, shuffle=False) # pongo shuffle flase ya que en el enunciado se piden esos resultados pero creo que deberia ser mejor con shuffle true
X_test, y_test = create_windows_np(test_data, window_size=past, horizon=future, shuffle=False)
print(X_train.shape, y_train.shape)
print(f'x[0]: {X_train[0]}, y[0]: {y_train[0]}')
print(f'x[1]: {X_train[1]}, y[1]: {y_train[1]}')
print(f'x[2]: {X_train[2]}, y[2]: {y_train[2]}')
print(f'x[3]: {X_train[3]}, y[3]: {y_train[3]}')

(2994, 5) (2994,)
x[0]: [20.7 17.9 18.8 14.6 15.8], y[0]: 15.8
x[1]: [17.9 18.8 14.6 15.8 15.8], y[1]: 17.4
x[2]: [18.8 14.6 15.8 15.8 15.8], y[2]: 21.8
x[3]: [14.6 15.8 15.8 15.8 17.4], y[3]: 20.0


<a name='3.2'></a>
## Cuestión 2: Cree un modelo recurrente de dos capas GRU para predecir con las ventanas de la cuestión anterior.


In [7]:
inputs = keras.layers.Input(shape=(past, 1))
gdr_1 = keras.layers.GRU(64, return_sequences=True)(inputs)
gdr_2 = keras.layers.GRU(64, return_sequences=False)(gdr_1)
outputs = keras.layers.Dense(1)(gdr_2)
model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=keras.optimizers.Adam(), loss="mse")
model.summary()

I0000 00:00:1794421295.787937   49308 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4597 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:01:00.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 5, 64)          │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,889 (148.00 KB)

 Trainable params: 37,889 (148.00 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200


I0000 00:00:1794421299.684921   49437 cuda_dnn.cc:442] Loaded cuDNN version 92801


38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 61ms/step - loss: 41.1924 - val_loss: 16.9724
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 15.9960 - val_loss: 16.3588
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 13.6166 - val_loss: 12.9472
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 11.2069 - val_loss: 10.4228
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9.8471 - val_loss: 9.4908
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9.2963 - val_loss: 8.9738
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 9.0877 - val_loss: 8.7387
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 8.9776 - val_loss: 8.6136
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 8.9108 - val_loss: 8.5266
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 8.8687 - val_loss: 8.4577
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 8.8417 - val_loss: 8.4015
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 8.8237

In [9]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 7.1647
Test Loss: 7.164728164672852


<a name='3.3'></a>
## Cuestión 3: Añada más features a la series temporal, por ejemplo `portion_year`. Cree un modelo que mejore al anterior.


In [10]:
## Puede añadir más features
df['portion_year'] = df['Date'].dt.dayofyear / 365.0
df_multi = df[['Temp', 'portion_year']].copy()

## train - test split
train_data = df_multi.iloc[:3000].copy()
test_data = df_multi.loc[3000:, :].copy()

In [11]:
## Create windows
def create_windows_multivariate_np(data, window_size, horizon, target_col_idx, shuffle=False):
    """
    Creates a dataset from the given time series data using NumPy.

    Parameters:
    data (np.ndarray or pd.DataFrame): Time series data with multiple features.
    window_size (int): The number of past time steps to use as input features.
    horizon (int): The number of future time steps to predict.
    target_col_idx (int): The index of the target column in the input data.
    shuffle (bool): Whether to shuffle the data or not.

    Returns:
    tuple: A tuple containing the input-output pairs (X, y) as NumPy arrays.
    """
    if isinstance(data, pd.DataFrame):
        data = data.values

    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i+window_size, :])
        y.append(data[i+window_size+horizon-1, target_col_idx])

    X, y = np.array(X), np.array(y)

    if shuffle:
        indices = np.arange(X.shape[0])
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]

    return X, y

X_train, y_train = create_windows_multivariate_np(train_data, window_size=past, horizon=future, target_col_idx= 0, shuffle=False)
X_test, y_test = create_windows_multivariate_np(test_data, window_size=past, horizon=future, target_col_idx= 0, shuffle=False)
print(X_train.shape, y_train.shape)

(2994, 5, 2) (2994,)


In [12]:
inputs = keras.layers.Input(shape=(past, 2))
gdr_1 = keras.layers.GRU(64, return_sequences=True)(inputs)
gdr_2 = keras.layers.GRU(64, return_sequences=False)(gdr_1)
outputs = keras.layers.Dense(1)(gdr_2)
model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=keras.optimizers.Adam(), loss="mse")
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 5, 2)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 5, 64)          │        13,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,081 (148.75 KB)

 Trainable params: 38,081 (148.75 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 47.8197 - val_loss: 20.2188
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 16.8619 - val_loss: 16.8022
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 15.7234 - val_loss: 16.0826
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 12.8521 - val_loss: 12.0382
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 10.6128 - val_loss: 10.0742
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9.6226 - val_loss: 9.2380
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9.2804 - val_loss: 8.8399
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9.1313 - val_loss: 8.6572
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9.0284 - val_loss: 8.5512
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 8.9532 - val_loss: 8.4801
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 8.9008 - val_loss: 8.4285
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 

In [14]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 6.9297
Test Loss: 6.92972993850708


<a name='3.4'></a>
## Cuestión 4: ¿En cuáles de estas aplicaciones se usaría un arquitectura 'many-to-one'?

**a)** Clasificación de sentimiento en textos

**b)** Verificación de voz para iniciar el ordenador.

**c)** Generación de música.

**d)** Un clasificador que clasifique piezas de música según su autor.


A, D

<a name='3.5'></a>
## Cuestión 5: ¿Qué ventajas aporta el uso de word embeddings?

**a)** Permiten reducir la dimensión de entrada respecto al one-hot encoding.

**b)** Permiten descubrir la similaridad entre palabras de manera más intuitiva que con one-hot encoding.

**c)** Son una manera de realizar transfer learning en nlp.

**d)** Permiten visualizar las relaciones entre palabras con métodos de reducción de dimensioones como el PCA.


A, B, D